# Final Validation Runner
Purpose: Validate the `day9-corpus-expansion-100` branch directly in Google Colab using the Google Drive `Data_v2/` bundle.

In [ ]:
# @title Configuration
DATA_PATH = "/content/drive/MyDrive/AIGC/Data_v2" # @param {type:"string"}
BRANCH = "day9-corpus-expansion-100" # @param {type:"string"}

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
%%bash -s "$BRANCH"
# Cell 2: Clone branch
BRANCH_NAME=$1
rm -rf /content/AIGC_Fake_Detection
git clone -b $BRANCH_NAME https://github.com/IanJ332/AIGC_Fake_Detection.git /content/AIGC_Fake_Detection
cd /content/AIGC_Fake_Detection
git branch --show-current
git log --oneline -3

In [ ]:
%%bash
# Cell 3: Install dependencies
cd /content/AIGC_Fake_Detection
pip install -r requirements.txt

In [ ]:
# Cell 4: Verify Drive Data
from pathlib import Path

DATA_DIR = Path(DATA_PATH)

required = [
    DATA_DIR / "extracted" / "entities.csv",
    DATA_DIR / "extracted" / "results.csv",
    DATA_DIR / "sections" / "sections.jsonl",
    DATA_DIR / "tables" / "table_candidates.jsonl",
    DATA_DIR / "registry" / "manifest_100.csv",
    DATA_DIR / "index" / "research_corpus.duckdb",
]

missing = []
for p in required:
    exists = p.exists()
    print(p, "=>", exists)
    if not exists:
        missing.append(str(p))

if missing:
    print("WARNING: Missing some data files. Full validation may fail.")
else:
    print("READY_TO_RUN_VALIDATION_ON_DRIVE")

In [ ]:
%%bash
# Cell 5: Compile checks
cd /content/AIGC_Fake_Detection
python -m py_compile src/**/*.py eval/*.py scripts/*.py

In [ ]:
%%bash -s "$DATA_PATH"
# Cell 6: Eval-only reproduction
DPATH=$1
cd /content/AIGC_Fake_Detection
export PYTHONPATH=/content/AIGC_Fake_Detection:$PYTHONPATH

python eval/run_eval.py \
  --data-dir $DPATH \
  --questions eval/questions_40.jsonl

In [ ]:
%%bash -s "$DATA_PATH"
# Cell 7: Budget eval
DPATH=$1
cd /content/AIGC_Fake_Detection
export PYTHONPATH=/content/AIGC_Fake_Detection:$PYTHONPATH

python eval/run_budget_eval.py \
  --data-dir $DPATH \
  --questions eval/questions_40.jsonl \
  --out-dir artifacts/reports

In [ ]:
%%bash -s "$DATA_PATH"
# Cell 8: Critical demo questions
DPATH=$1
cd /content/AIGC_Fake_Detection
export PYTHONPATH=/content/AIGC_Fake_Detection:$PYTHONPATH

echo "===== Citation Graph Demo ====="
python -m src.query.cli \
  --data-dir $DPATH \
  --question "Which papers in the corpus build on the GenImage benchmark?"

echo "===== Multihop Demo ====="
python -m src.query.cli \
  --data-dir $DPATH \
  --question "Find papers that use both the CLIP model and the Accuracy metric."

echo "===== Quantitative Demo ====="
python -m src.query.cli \
  --data-dir $DPATH \
  --question "How many papers mention GenImage?"

In [ ]:
%%bash -s "$DATA_PATH"
# Cell 9: Sync validation outputs to Drive
DPATH=$1
mkdir -p $DPATH/reports/day9_eval

cp -r /content/AIGC_Fake_Detection/eval/results/* \
  $DPATH/reports/day9_eval/ 2>/dev/null || true

cp /content/AIGC_Fake_Detection/artifacts/reports/budget_eval_results.csv \
  $DPATH/reports/day9_eval/ 2>/dev/null || true

cp /content/AIGC_Fake_Detection/artifacts/reports/budget_eval_summary.md \
  $DPATH/reports/day9_eval/ 2>/dev/null || true

echo "Saved validation outputs to Drive:"
ls -lah $DPATH/reports/day9_eval

In [ ]:
# Cell 10: Final status summary
from pathlib import Path

drive_report_dir = Path(DATA_PATH) / "reports" / "day9_eval"
repo_report_dir = Path("/content/AIGC_Fake_Detection/artifacts/reports")

checks = {
    "day6_eval_summary": (drive_report_dir / "day6_eval_summary.md").exists() or (repo_report_dir / "day6_eval_summary.md").exists(),
    "budget_eval_summary": (drive_report_dir / "budget_eval_summary.md").exists() or (repo_report_dir / "budget_eval_summary.md").exists(),
    "budget_eval_results": (drive_report_dir / "budget_eval_results.csv").exists() or (repo_report_dir / "budget_eval_results.csv").exists(),
}

for k, v in checks.items():
    print(k, "=>", v)

if all(checks.values()):
    print("FINAL_VALIDATION_PASS")
else:
    print("FINAL_VALIDATION_CAUTION")